In [4]:
import os
import numpy as np
import cv2

from matplotlib import pyplot as plt
import matplotlib 

# Local descriptors
from skimage.feature import hog
from skimage import  exposure
from skimage import feature

from time import time
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import MinMaxScaler

In [ ]:
import os
import pickle
import numpy as np
from deepface import DeepFace
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix


def get_image_paths_by_class(dataset_folder):
    X_paths = []
    y_labels = []
    for class_name in sorted(os.listdir(dataset_folder)):
        class_dir = os.path.join(dataset_folder, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            X_paths.append(os.path.join(class_dir, fname))
            y_labels.append(class_name)
    return X_paths, y_labels


def compute_embeddings(image_paths, model_name="Facenet"):
    embeddings = []
    valid_labels_idx = []  # índices de las imágenes que sí se han podido procesar
    for idx, path in enumerate(image_paths):
        try:
            emb_obj = DeepFace.represent(
                img_path=path,
                model_name=model_name,
                enforce_detection=False  # asumimos que hay cara, pero no queremos que casque
            )
            embeddings.append(emb_obj[0]["embedding"])
            valid_labels_idx.append(idx)
        except Exception as e:
            print(f"[AVISO] No se pudo procesar {path}: {e}")
    if len(embeddings) == 0:
        raise RuntimeError("No se pudo obtener ningún embedding. Revisa el dataset.")
    return np.asarray(embeddings, dtype="float32"), valid_labels_idx


def train_emotion_svm(train_folder, model_output_path, n_splits=5, model_name="Facenet"):
    # 1) Cargar rutas e etiquetas
    X_paths, y_labels = get_image_paths_by_class(train_folder)
    if len(X_paths) == 0:
        raise RuntimeError("El dataset de entrenamiento está vacío o las rutas son incorrectas.")

    # 2) Embeddings
    print("[INFO] Calculando embeddings de entrenamiento con DeepFace...")
    X, valid_idx = compute_embeddings(X_paths, model_name=model_name)
    y_labels = [y_labels[i] for i in valid_idx]

    # 3) Codificar etiquetas
    le = LabelEncoder()
    y = le.fit_transform(y_labels)

    # 4) Escalado + SVM
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clf = SVC(kernel="rbf", probability=True, class_weight="balanced")

    # 5) Validación cruzada
    print("[INFO] Validación cruzada (StratifiedKFold)...")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    precs = []
    recs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y), start=1):
        X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_val)

        prec = precision_score(y_val, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_val, y_pred, average="weighted", zero_division=0)
        precs.append(prec)
        recs.append(rec)

        print(f"\n=== Fold {fold} ===")
        print(f"Precisión (weighted): {prec:.3f}")
        print(f"Recall    (weighted): {rec:.3f}")
        print("\nClassification report:")
        print(classification_report(y_val, y_pred, target_names=le.classes_, zero_division=0))
        print("Matriz de confusión:")
        print(confusion_matrix(y_val, y_pred, labels=range(len(le.classes_))))

    print("\n=== Medias sobre los folds (entrenamiento) ===")
    print(f"Precisión media: {np.mean(precs):.3f}")
    print(f"Recall medio:    {np.mean(recs):.3f}")

    # 6) Entrenar modelo final con todos los datos de entrenamiento
    print("\n[INFO] Entrenando modelo final con todo el conjunto de entrenamiento...")
    clf.fit(X_scaled, y)

    # 7) Guardar a disco
    model = {
        "scaler": scaler,
        "label_encoder": le,
        "classifier": clf,
        "deepface_model_name": model_name
    }
    with open(model_output_path, "wb") as f:
        pickle.dump(model, f)

    print(f"[INFO] Modelo guardado en: {model_output_path}")


def evaluate_on_folder(test_folder, model_path):
    # 1) Cargar modelo
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    scaler = model["scaler"]
    le = model["label_encoder"]
    clf = model["classifier"]
    model_name = model.get("deepface_model_name", "Facenet")

    # 2) Cargar rutas e etiquetas reales
    X_paths, y_labels = get_image_paths_by_class(test_folder)
    if len(X_paths) == 0:
        raise RuntimeError("El dataset de test está vacío o las rutas son incorrectas.")

    print("[INFO] Calculando embeddings de test con DeepFace...")
    X_test, valid_idx = compute_embeddings(X_paths, model_name=model_name)
    y_labels = [y_labels[i] for i in valid_idx]
    y_true = le.transform(y_labels)

    # 3) Escalar y predecir
    X_test_scaled = scaler.transform(X_test)
    y_pred = clf.predict(X_test_scaled)

    # 4) Métricas
    print("\n=== Evaluación en el conjunto de test ===")
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
    print("Matriz de confusión:")
    print(confusion_matrix(y_true, y_pred, labels=range(len(le.classes_))))


if __name__ == "__main__":
    DATA_ROOT = r"C:\Users\luisp\Desktop\VC\emotions"

    TRAIN_DIR = os.path.join(DATA_ROOT, "train")  # aquí entrenas
    TEST_DIR  = os.path.join(DATA_ROOT, "test")        # aquí evalúas

    MODEL_PATH = "modelo_emociones_svm.pkl"

    # Entrenar
    train_emotion_svm(TRAIN_DIR, MODEL_PATH, n_splits=5, model_name="Facenet")

    # Evaluar
    evaluate_on_folder(TEST_DIR, MODEL_PATH)


[INFO] Calculando embeddings de entrenamiento con DeepFace...


In [ ]:
import cv2
import numpy as np
from deepface import DeepFace
import pickle

MODEL_PATH = "modelo_emociones_svm.pkl"

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

scaler = model["scaler"]
label_encoder = model["label_encoder"]
classifier = model["classifier"]
deepface_model_name = model.get("deepface_model_name", "Facenet")


def apply_emotion_filter(frame, emotion):
    """Aplica un filtro sencillo según la emoción detectada."""
    emotion = emotion.lower()

    if emotion == "happy":
        # Aumentar saturación (más color)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hsv[..., 1] = np.clip(hsv[..., 1] * 1.5, 0, 255).astype(np.uint8)
        filtered = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    elif emotion == "sad":
        # Escala de grises con tono azulado
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        blue_overlay = np.full_like(gray, (255, 0, 0))  # BGR: azul
        filtered = cv2.addWeighted(gray, 0.7, blue_overlay, 0.3, 0)

    elif emotion == "angry":
        # Tinte rojizo
        red_overlay = np.full_like(frame, (0, 0, 255))  # BGR: rojo
        filtered = cv2.addWeighted(frame, 0.6, red_overlay, 0.4, 0)

    elif emotion in ("surprise", "surprised"):
        # Aumentar brillo y contraste
        filtered = cv2.convertScaleAbs(frame, alpha=1.3, beta=25)

    else:
        # Emociones neutras / desconocidas: filtro suave sin cambios fuertes
        filtered = cv2.GaussianBlur(frame, (7, 7), 0)

    return filtered


def run_emotion_filter_demo():
    cap = cv2.VideoCapture(1)
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la webcam.")

    print("[INFO] Pulsar ESC para salir.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        try:
    
            emb_obj = DeepFace.represent(
                img_path=frame,
                model_name=deepface_model_name,
                enforce_detection=True  # lanza excepción si no hay cara
            )

            embedding = np.array(emb_obj[0]["embedding"], dtype="float32").reshape(1, -1)

            embedding_scaled = scaler.transform(embedding)
            y_pred = classifier.predict(embedding_scaled)
            emotion = label_encoder.inverse_transform(y_pred)[0]

            filtered = apply_emotion_filter(frame, emotion)

            txt = f"Emocion (SVM): {emotion}"
            cv2.putText(filtered, txt, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

        except Exception:
            # Si no detecta cara o hay cualquier problema
            filtered = frame.copy()
            cv2.putText(filtered, "Sin cara / no detectada", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        cv2.imshow("Prototipo 1 - Filtros por emocion (modelo propio)", filtered)
        if cv2.waitKey(1) & 0xFF == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()


# EJEMPLO DE USO:
run_emotion_filter_demo()



[INFO] Pulsar ESC para salir.


In [3]:
import cv2
import numpy as np
from deepface import DeepFace 
import pygame

def overlay_on_face(frame, region, overlay, scale=1.4):
    x = region["x"]
    y = region["y"]
    w = int(region["w"] * scale)
    h = int(region["h"] * scale)
    x = x - (w - region["w"]) // 2
    y = y - (h - region["h"]) // 2
    x = max(0, x)
    y = max(0, y)
    h_frame, w_frame = frame.shape[:2]
    w = min(w, w_frame - x)
    h = min(h, h_frame - y)
    if w <= 0 or h <= 0:
        return frame

    overlay_resized = cv2.resize(overlay, (w, h))
    if overlay_resized.shape[2] == 4:
        b, g, r, a = cv2.split(overlay_resized)
        overlay_bgr = cv2.merge((b, g, r))
        alpha = a.astype(float) / 255.0
        alpha = np.stack([alpha, alpha, alpha], axis=-1)
        roi = frame[y:y + h, x:x + w].astype(float)
        blended = alpha * overlay_bgr.astype(float) + (1 - alpha) * roi
        frame[y:y + h, x:x + w] = blended.astype(np.uint8)
    else:
        frame[y:y + h, x:x + w] = overlay_resized

    return frame

def run_face_filter_demo():
    pygame.mixer.init()
    pygame.mixer.music.load("./audio/torero.mp3")

    overlay = cv2.imread("./images/chayanne.webp", cv2.IMREAD_UNCHANGED)
    if overlay is None:
        raise RuntimeError("No se pudo cargar ./images/chayanne.webp")

    cap = cv2.VideoCapture(1)
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la webcam.")

    sonido_activo = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cara_detectada = False

        try:
            obj = DeepFace.analyze(
                img_path=frame,
                actions=["emotion"],
                enforce_detection=True
            )

            if isinstance(obj, list):
                regions = [item["region"] for item in obj]
            else:
                regions = [obj["region"]]

            cara_detectada = True

            filtered = frame.copy()
            for region in regions:
                filtered = overlay_on_face(filtered, region, overlay, scale=1.4)

        except Exception:
            filtered = frame.copy()

        if cara_detectada and not sonido_activo:
            pygame.mixer.music.play()
            sonido_activo = True
        elif not cara_detectada and sonido_activo:
            pygame.mixer.music.stop()
            sonido_activo = False

        cv2.imshow("Filtro Chayanne", filtered)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    pygame.mixer.quit()

run_face_filter_demo()


pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
